# xrf-raw-to-h5 — demo: CLI + UI

I tre percorsi d'uso della stessa pipeline di conversione (lettura → calibrazione →
rebin opzionale → writer): **CLI** (quella che lanceresti da terminale),
**API Python** (`xrfraw.io.convert_file`) e **UI embeddata** (Dash interattiva).

Prerequisito: eseguire dentro la venv del progetto (`uv sync --dev`).
Se `data_example/mappa11.txt` è presente viene usato; altrimenti viene
generato un piccolo file sintetico (formato B, griglia 3×2, 8 eventi).
Gli output finiscono in `demo_out/` (gitignored).

In [ ]:
import subprocess
import sys
from pathlib import Path

# Se esiste il dato d'esempio reale lo usiamo; altrimenti file sintetico piccolo (formato B)
RAW = Path("data_example/mappa11.txt")
if not RAW.is_file():
    RAW = Path("demo_tiny.txt")
    RAW.write_text("\n".join([
        # serpentina 3x2: X 0..2000 step 1000, Y 0..5000 step 5000, 8 eventi
        "50000000", "60000000", "0", "5", "5", "7",
        "50001000", "60000000",
        "50002000", "60000000", "0",
        "50000000", "60005000",
        "50001000", "60005000", "12",
        "50002000", "60005000", "12", "12",
        "</analysisData>", "</XRFanalysis>", "",
    ]))
OUT = Path("demo_out")
OUT.mkdir(exist_ok=True)
print(f"raw di demo: {RAW} ({RAW.stat().st_size / 1e6:.2f} MB)")
print(f"python: {sys.executable}")

## 1. CLI

La stessa identica command line del terminale: `xrf-raw-to-h5 INPUT -o OUT_DIR`.
Il notebook la lancia tramite `python -m xrfraw` (lo script console
`xrf-raw-to-h5` è il suo alias).

Opzioni: `--format h5|npz`, `--n-channels N` (default 16384),
`--dtype uint32|uint16`, `--metadata JSON` (griglia/calibrazione per formato B),
`--rebin SIZE sum|mean`, `--overwrite` (default: rifiuta di sovrascrivere).
Exit code: 0 ok, 1 conversione fallita, 2 errore d'uso.

In [ ]:
# stessa command line del terminale
cmd = [sys.executable, "-m", "xrfraw", str(RAW),
       "-o", str(OUT), "--format", "h5", "--overwrite"]
print("$", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout.strip())
print(r.stderr.strip())
assert r.returncode == 0, "CLI fallita"
out_file = OUT / f"{RAW.stem}.h5"
print(f"output: {out_file} ({out_file.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import json

import h5py

with h5py.File(out_file, "r") as h:
    img  = h["img"][...]
    meta = json.loads(h.attrs["metadata_json"])   # metadata_json è un attr della root

print("shape:", img.shape, img.dtype)   # (ny, nx, n_channels)
print("n_events:", meta["n_events"], "| sum(img):", int(img.sum()),
      "| coerenti:", int(img.sum()) == meta["n_events"])
print("anomalies:", meta.get("anomalies"))

## 2. API Python

Stessa pipeline senza terminale: `xrfraw.io.convert_file` è la funzione
che CLI e UI condividono. Qui viene chiamata esattamente come fa la
CLI sopra (h5, uint32, NC default, overwrite) e restituisce il `Dataset`
scritto: `ds.img` (y, x, canali) e `ds.metadata`.
Tutti i parametri sono scritti esplicitamente con il loro default:
è la lista completa di ciò che si può cambiare.


In [ ]:
from xrfraw.io import convert_file

# tutti i parametri espliciti (nome=valore): è la lista completa di ciò che possiamo cambiare
ds = convert_file(
    src=RAW,                   # input raw (formato A o B)
    out_path=OUT / f"{RAW.stem}.h5",   # output (stesso file della CLI)
    fmt="h5",                 # "h5" | "npz"
    n_channels=None,           # None = default strumentale (16384)
    dtype="uint32",           # "uint32" | "uint16"
    metadata=None,             # formato B: dict griglia/calibrazione
    rebin_spec=None,           # es. (2, "sum") o (2, "mean"); None = off
    cal=None,                  # None = calibrazione dichiarata nell'XML; oppure {"m": ..., "q": ...}
    overwrite=True,            # False = rifiuta di sovrascrivere un file esistente
)
print("shape:", ds.img.shape, ds.img.dtype)
print("n_events:", ds.metadata["n_events"], "| sum(img):", int(ds.img.sum()),
      "| coerenti:", int(ds.img.sum()) == ds.metadata["n_events"])

## 3. UI embeddata

La stessa pipeline, interattiva: carichi il raw (formato A o B), vedi report
di validazione e metadati, imposti m/q (provenance `user`) se vuoi, scegli
dtype/rebin/formato e scarichi l'output. Utile per esplorare i file prima di
lanciarli in batch via CLI.

In [ ]:
from xrfraw.ui import build_app

app = build_app()
app.run(
    jupyter_mode="inline", 
    debug=False, 
    jupyter_height=800
)  # da terminale: app.run()

## Note

- CLI: scrive `demo_out/<nome-input>.h5`; UI: restituisce il file come
  download (stesso writer, stessa pipeline condivisa `io.convert_file`).
- Calibrazione: applicata automaticamente se l'XML la dichiara (formato A);
  per il formato B si imposta dalla UI o con `--metadata`.
- Rebin: sempre esplicito, off di default, dichiarato nei metadati dell'output.
- Senza `--overwrite` la CLI rifiuta di sovrascrivere (stesso comportamento
  nel writer: nessun overwrite silenzioso).